In [ ]:
# BLOCK 1: IMPORT LIBRARIES AND LOAD DATA

import pandas as pd
import numpy as np
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("ted_talks_final_cleaned.csv")

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (1202, 32)


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,recorded_year,published_year,topic_count,title_word_count,description_word_count,transcript_word_count,about_speaker_word_count,publication_year,related_talk_count,combined_text
0,1,averting the climate crisis,al gore,Al Gore,climate advocate,nobel laureate al gore focused the world’s att...,3523392,2006-02-25,2006-06-27,TED2006,...,2006.0,2006.0,12.0,4.0,38.0,2148.0,23.0,2006.0,0.0,averting the climate crisis alternative energy...
1,92,the best stats youve ever seen,hans rosling,Hans Rosling,global health expert data visionary,in hans rosling’s hands data sings global tren...,14501685,2006-02-22,2006-06-27,TED2006,...,2006.0,2006.0,13.0,6.0,26.0,3140.0,32.0,2006.0,0.0,the best stats youve ever seen Africa Asia Goo...
2,7,simplicity sells,david pogue,David Pogue,technology columnist,david pogue is the personal technology columni...,1920832,2006-02-24,2006-06-27,TED2006,...,2006.0,2006.0,10.0,2.0,31.0,3345.0,43.0,2006.0,0.0,simplicity sells computers entertainment inter...
3,53,greening the ghetto,majora carter,Majora Carter,activist for environmental justice,majora carter redefined the field of environme...,2664069,2006-02-26,2006-06-27,TED2006,...,2006.0,2006.0,10.0,3.0,30.0,3052.0,31.0,2006.0,0.0,greening the ghetto MacArthur grant activism b...
4,66,do schools kill creativity,sir ken robinson,Sir Ken Robinson,author educator,creativity expert sir ken robinson challenges ...,65051954,2006-02-25,2006-06-27,TED2006,...,2006.0,2006.0,7.0,4.0,21.0,3158.0,30.0,2006.0,0.0,do schools kill creativity children creativity...


In [ ]:
# BLOCK 2: SELECT RELEVANT COLUMNS

relevant_columns = [
    "title",
    "topics",
    "description",
    "transcript"
]

data = df[relevant_columns].copy()

print("Selected Columns:")
print(data.columns.tolist())

display(data.head())

Selected Columns:
['title', 'topics', 'description', 'transcript']


,title,topics,description,transcript
0,averting the climate crisis,alternative energy cars climate change culture...,with the same humor and humanity he exuded in ...,thank you so much chris and its truly a great ...
1,the best stats youve ever seen,Africa Asia Google demo economics global issue...,youve never seen data presented like this with...,about years ago i took on the task to teach gl...
2,simplicity sells,computers entertainment interface design media...,new york times columnist david pogue takes aim...,music the sound of silence simon garfunkel hel...
3,greening the ghetto,MacArthur grant activism business cities envir...,in an emotionally charged talk macarthurwinnin...,if youre here today — and im very happy that y...
4,do schools kill creativity,children creativity culture dance education pa...,sir ken robinson makes an entertaining and pro...,good morning how are you audience good its bee...


In [ ]:
# BLOCK 3: HANDLE MISSING VALUES

# Title is essential
data = data.dropna(subset=["title"])

# Replace missing values in other text columns
for col in ["topics", "description", "transcript"]:
    data[col] = data[col].fillna("")

print("Missing values after handling:")

display(data.isnull().sum())

Missing values after handling:


,0
title,0
topics,0
description,0
transcript,0


In [ ]:
# BLOCK 4: TEXT PREPROCESSING

def preprocess_text(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove HTML
    text = re.sub(r"<.*?>", "", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in [
    "title",
    "topics",
    "description",
    "transcript"
]:
    data[col] = data[col].apply(preprocess_text)

print("Text preprocessing completed.")

Text preprocessing completed.


In [ ]:
# BLOCK 5: CREATE COMBINED TEXT

data["combined_text"] = (
    data["title"] + " " +
    data["topics"] + " " +
    data["description"] + " " +
    data["transcript"]
)

data["combined_text"] = (
    data["combined_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

display(
    data[
        ["title", "combined_text"]
    ].head()
)

,title,combined_text
0,averting the climate crisis,averting the climate crisis alternative energy...
1,the best stats youve ever seen,the best stats youve ever seen africa asia goo...
2,simplicity sells,simplicity sells computers entertainment inter...
3,greening the ghetto,greening the ghetto macarthur grant activism b...
4,do schools kill creativity,do schools kill creativity children creativity...


In [ ]:
# BLOCK 6: TF-IDF VECTORIZATION

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

tfidf_matrix = tfidf.fit_transform(
    data["combined_text"]
)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Number of Features:", len(tfidf.get_feature_names_out()))

TF-IDF Matrix Shape: (63, 10000)
Number of Features: 10000


In [ ]:
# BLOCK 7: COSINE SIMILARITY

cosine_sim = cosine_similarity(tfidf_matrix)

print("Cosine Similarity Matrix Shape:",
      cosine_sim.shape)

Cosine Similarity Matrix Shape: (63, 63)


In [ ]:
# BLOCK 8: RECOMMENDATION FUNCTION

indices = pd.Series(
    data.index,
    index=data["title"]
).drop_duplicates()


def recommend_talks(title, n=10):

    if title not in indices:
        print("Talk not found in dataset.")
        return

    idx = indices[title]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected talk itself
    similarity_scores = similarity_scores[1:n+1]

    talk_indices = [
        i[0] for i in similarity_scores
    ]

    recommendations = data.iloc[
        talk_indices
    ][["title"]].copy()

    recommendations["Similarity Score"] = [
        round(i[1], 4)
        for i in similarity_scores
    ]

    return recommendations

In [ ]:
# BLOCK 9: TEST RECOMMENDATION

sample_talk = data["title"].iloc[0]

print("Selected TED Talk:")
print(sample_talk)

print("\nRecommended TED Talks:")

recommend_talks(
    sample_talk,
    n=10
)

Selected TED Talk:
averting the climate crisis

Recommended TED Talks:


,title,Similarity Score
20,nerdcore comedy,0.2183
2,simplicity sells,0.1965
50,global priorities bigger than climate change,0.1805
4,do schools kill creativity,0.1753
40,my wish three actions for africa,0.1645
11,my wish a call for opensource architecture,0.1612
9,my wish a global day of film,0.1570
21,meet the founder of the blog revolution,0.1529
6,letting go of god,0.1472
8,a life of purpose,0.1471


In [ ]:
# BLOCK 10: USER INPUT RECOMMENDATION

user_title = input(
    "Enter a TED Talk title: "
)

recommendations = recommend_talks(
    user_title,
    n=10
)

display(recommendations)

Enter a TED Talk title: a life of purpose


,title,Similarity Score
7,lets teach religion all religion in schools,0.2999
6,letting go of god,0.2591
57,four american characters,0.2304
4,do schools kill creativity,0.2087
38,my wish manufactured landscapes and green educ...,0.1969
40,my wish three actions for africa,0.1939
21,meet the founder of the blog revolution,0.1917
54,progress is not a zerosum game,0.1802
20,nerdcore comedy,0.1793
60,in praise of slowness,0.1775
